# Pipeline de datos · PhysioNet/CinC Challenge 2012

Lee los registros originales de set-a y set-b, limpia los valores imposibles y guarda todo en un único CSV en formato largo: una fila por medición, con los datos estáticos del paciente, el desenlace y la partición (train/val/test) repetidos en cada fila.

| Columnas | Contenido |
|---|---|
| `RecordID`, `Set`, `Split` | identificación del paciente y partición |
| `Age`, `Gender`, `Height`, `Weight`, `ICUType` | descriptores estáticos (vacío si faltan) |
| `In-hospital_death` | variable objetivo (0/1) |
| `SAPS-I`, `SOFA` | scores clínicos de referencia (vacío si faltan) |
| `Time` | horas desde el ingreso en la UCI |
| `Variable`, `Value` | la medición en sí |

Los pacientes sin ninguna medición aparecen con una sola fila y `Time`/`Variable`/`Value` vacíos, para que no se pierda ninguno.

## Configuración

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split



RAW = Path.cwd().parent /"data" / "raw"
SETS = {
    "a": {"records": RAW / "Train_Val/set-a/set-a", "outcomes": RAW / "Train_Val/Outcomes-a.txt"},
    "b": {"records": RAW / "Test/set-b/set-b",      "outcomes": RAW / "Test/Outcomes-b.txt"},
}
OUTPUT_CSV = Path("data/processed/physionet2012.csv")   

SEED = 42
VAL_FRACTION = 0.2          # parte de set-a que se reserva para validación

STATIC_VARS = ["Age", "Gender", "Height", "ICUType", "Weight"]

# Rangos fisiológicamente plausibles (inclusivos). Fuera de ellos, el valor se descarta.
VALID_RANGES = {
    # Constantes vitales, estado neurológico y balance hídrico
    "HR": (1, 300), "RespRate": (1, 100), "Temp": (25, 45), "GCS": (3, 15),
    "SysABP": (1, 300), "DiasABP": (1, 300), "MAP": (1, 300),
    "NISysABP": (1, 300), "NIDiasABP": (1, 300), "NIMAP": (1, 300),
    "Urine": (0, 10000), "Weight": (20, 300),
    # Soporte respiratorio y gasometría
    "MechVent": (1, 1), "FiO2": (0.21, 1.0), "SaO2": (1, 100),
    "PaO2": (1, 700), "PaCO2": (5, 150), "pH": (6.5, 8.0),
    # Laboratorio
    "Albumin": (0.5, 7), "ALP": (1, 5000), "ALT": (1, 20000), "AST": (1, 30000),
    "Bilirubin": (0.1, 80), "BUN": (1, 300), "Cholesterol": (20, 1000), "Creatinine": (0.1, 30),
    "Glucose": (10, 2000), "HCO3": (1, 60), "HCT": (5, 80), "K": (1, 15), "Lactate": (0.1, 40),
    "Mg": (0.1, 20), "Na": (90, 200), "Platelets": (1, 2000), "TroponinI": (0.01, 100),
    "TroponinT": (0.001, 100), "WBC": (0.1, 300),
}
STATIC_RANGES = {
    "Age": (15, 100), "Gender": (0, 1), "Height": (100, 230), "ICUType": (1, 4), "Weight": (20, 300),
}

## 1. Lectura de los registros

In [2]:
def read_set(records_dir):
    """Junta todos los .txt de un set en una tabla (RecordID, Time, Parameter, Value)."""
    records = []
    for path in Path(records_dir).glob("*.txt"):
        record = pd.read_csv(path)
        id = record.loc[record["Parameter"] == "RecordID", "Value"].values[0]
        record["RecordID"] = id
        record['Time'] = record['Time'].str.split(":").apply(lambda x: int(x[0]) + int(x[1])/60)
        records.append(record)
    return pd.concat(records)


def split_static_and_series(raw):
    """Separa las variables estáticas de las series temporales."""

    at_admission = raw[raw["Parameter"].isin(STATIC_VARS)]
    descriptors = at_admission.drop_duplicates(["RecordID", "Parameter"])
    static_df = descriptors.pivot(index="RecordID", columns="Parameter", values="Value")
    # Series temporales: quitar estáticas y RecordID
    series = raw[
                (~raw["Parameter"].isin(STATIC_VARS)) &
                (raw["Parameter"] != "RecordID") &
                raw["Value"].notna()].copy()

    series = series.rename(columns={"Parameter": "Variable"})[["RecordID", "Time", "Variable", "Value"]]

    return static_df, series


## 2. Limpieza

In [3]:
def clean_series(series):
    """Corrige el pH escrito sin punto decimal (735 -> 7.35) y descarta valores imposibles."""
    series = series.copy()

    ph_without_decimal = (series["Variable"] == "pH") & series["Value"].between(650, 800)
    series.loc[ph_without_decimal, "Value"] /= 100

    lower = series["Variable"].map({var: lo for var, (lo, hi) in VALID_RANGES.items()})
    upper = series["Variable"].map({var: hi for var, (lo, hi) in VALID_RANGES.items()})
    plausible = series["Value"].between(lower, upper)

    discarded = series.loc[~plausible, "Variable"].value_counts()
    return series[plausible], discarded


def clean_static(static):
    """Los -1 (dato ausente) y los valores imposibles pasan a NaN."""
    static = static.copy()
    for var, (lo, hi) in STATIC_RANGES.items():
        static[var] = static[var].where(static[var].between(lo, hi))
    return static

## 3. Desenlace

In [4]:
def read_outcomes(path):
    """Variable objetivo y scores clínicos. En los scores, -1 significa 'no disponible'."""
    outcomes = pd.read_csv(path)
    outcomes = outcomes[["RecordID","In-hospital_death", "SAPS-I", "SOFA"]]
    outcomes[["SAPS-I", "SOFA"]] = outcomes[["SAPS-I", "SOFA"]].where(outcomes[["SAPS-I", "SOFA"]] >= 0)
    return outcomes

## 4. Procesar cada set

In [5]:

patients, observations = [], []

for set_name, paths in SETS.items():
   static, series = split_static_and_series(read_set(paths["records"]))

   series, discarded = clean_series(series)
   series["Set"] = set_name
   series = series.set_index(["RecordID", "Set"])

   static = clean_static(static)
   outcomes = read_outcomes(paths["outcomes"])
   patient_info = static.merge(outcomes, left_index=True, right_on="RecordID", how="left")
   patient_info["Set"] = set_name
   patient_info = patient_info.set_index(["RecordID", "Set"])

   patients.append(patient_info)
   observations.append(series)

patients = pd.concat(patients)
series = pd.concat(observations)

In [6]:
display(patients.head())
display(series.head())

,,Age,Gender,Height,ICUType,Weight,In-hospital_death,SAPS-I,SOFA
RecordID,Set,,,,,,,,
132539,a,54.0,0.0,NaN,4.0,NaN,0,6.0,1.0
132540,a,76.0,1.0,175.3,2.0,76.0,0,16.0,8.0
132541,a,44.0,0.0,NaN,3.0,56.7,0,21.0,11.0
132543,a,68.0,1.0,180.3,3.0,84.6,0,7.0,1.0
132545,a,88.0,0.0,NaN,3.0,NaN,0,17.0,2.0


Time   Variable   Value
RecordID Set                             
132592.0 a    1.333333        GCS   15.00
         a    1.333333         HR  112.00
         a    1.333333  NIDiasABP   43.00
         a    1.333333      NIMAP   68.67
         a    1.333333   NISysABP  120.00

## 5. Particiones: train / val salen de set-a (estratificado), test = set-b

In [7]:
ids_a = patients.index[patients.index.get_level_values("Set") == "a"]

ids_train, ids_val = train_test_split(ids_a, test_size=VAL_FRACTION, stratify=patients.loc[ids_a, "In-hospital_death"], random_state=SEED)

patients["Split"] = "test"
patients.loc[ids_train, "Split"] = "train"
patients.loc[ids_val, "Split"] = "val"


## 6. Un único CSV

In [8]:
columns = ["RecordID", "Set", "Split", *STATIC_VARS, "In-hospital_death", "SAPS-I", "SOFA", "Time", "Variable", "Value"]

dataset = (patients.reset_index()
           .merge(series.reset_index(), on=["RecordID", "Set"], how="left")
           .sort_values(["Set", "RecordID", "Time", "Variable"], kind="stable", na_position="first")
           [columns])

integer_columns = ["Gender", "ICUType", "In-hospital_death", "SAPS-I", "SOFA"]
dataset[integer_columns] = dataset[integer_columns].astype("Int64")
dataset = dataset.dropna(subset=['Variable'])
dataset["Time"] = dataset["Time"].round(4)

dataset.to_csv(OUTPUT_CSV, index=False)

print(f"Guardado {OUTPUT_CSV}: {len(dataset):,} filas, {dataset[['RecordID', 'Set']].drop_duplicates().shape[0]} pacientes")

OSError: Cannot save file into a non-existent directory: 'data/processed'

In [ ]:
dataset

,RecordID,Set,Split,Age,Gender,Height,ICUType,Weight,In-hospital_death,SAPS-I,SOFA,Time,Variable,Value
0,132539,a,train,54.0,0,NaN,4,NaN,0,6,1,0.1167,GCS,15.00
1,132539,a,train,54.0,0,NaN,4,NaN,0,6,1,0.1167,HR,73.00
2,132539,a,train,54.0,0,NaN,4,NaN,0,6,1,0.1167,NIDiasABP,65.00
3,132539,a,train,54.0,0,NaN,4,NaN,0,6,1,0.1167,NIMAP,92.33
4,132539,a,train,54.0,0,NaN,4,NaN,0,6,1,0.1167,NISysABP,147.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218389,152864,b,test,82.0,0,NaN,2,NaN,0,18,8,47.2000,DiasABP,40.00
3218390,152864,b,test,82.0,0,NaN,2,NaN,0,18,8,47.2000,HR,63.00
3218391,152864,b,test,82.0,0,NaN,2,NaN,0,18,8,47.2000,MAP,68.00
3218392,152864,b,test,82.0,0,NaN,2,NaN,0,18,8,47.2000,SysABP,124.00


: 